# IDX Exchange Data Science Internship - Week Seven - Advanced Models

* Try Gradient Boosting (e.g., XGBoost or LightGBM)
* Perform light hyperparameter tuning (depth, learning rate, n_estimators)

--------

In [12]:
!pip install xgboost scikit-learn pandas

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import xgboost as xgb
from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import PredefinedSplit

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error

This week we will try to push our predictions even further by introducing gradient boosting to the model. In particular, we'll be using XGBoost, which sequentially trains decision trees to minimize a loss function (mean squared error, in this case). For datasets like ours (clean, moderately large), we can expect XGBoost to perform better than Random Forests.

## Import Data

In [2]:
# Read in the training and test sets created last two

train_and_val = pd.read_csv("week_3_train_2.csv")
testing = pd.read_csv("week_3_test_2.csv")

# Carve out training and validation sets
training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265326, 47) (255033, 47) (10293, 47)


In [3]:
common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc"]

x_train = training[common_cols]  
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [2]:
def median_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true))

def metrics(y_true, y_pred, logged=True):
    # Note: you have to undo the log transform on ClosePrice by using np.exp()
    if logged == True:
        y_true = np.exp(y_true)
        y_pred = np.exp(y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mdape = median_absolute_percentage_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mape * 100, mdape * 100, mae

def metric_df(r2_train, mape_train, mdape_train, mae_train,
              r2_val, mape_val, mdape_val, mae_val,
              r2_test, mape_test, mdape_test, mae_test):
    data = {"r2": [r2_train, r2_val, r2_test],
            "MAPE": [mape_train, mape_val, mape_test],
            "MdAPE": [mdape_train, mdape_val, mdape_test],
            "MAE": [f"${round(mae_train, 2)}", f"${round(mae_val, 2)}", f"${round(mae_test, 2)}"]}
    df = pd.DataFrame(data)
    df.index = ["Train Set", "Validation Set", "Test Set"]
    return df

## Model One: Baseline Dataset

Let's first test XGBoost on the baseline, pre-feature engineering dataset. Since XGBoost has many hyperparameters to tune, we'll use GridSearch to test a few different configurations at a time, and then select the best option.

In [5]:
# In its default state, GridSearch takes a training set and performs cross validation
# Since we have a pre-determined validation set, we'll instead pass it in as a custom split

# Recombine the training and validation sets
x_combined = np.vstack((x_train, x_val))
y_combined = np.concatenate((y_train, y_val))

# Mark which indices are train and validation
split_indices = np.zeros(x_combined.shape[0])
split_indices[:len(x_train)] = -1 
split_indices[len(x_train):] = 0

# Create the custom split
pds = PredefinedSplit(test_fold=split_indices)

In [6]:
xgb_regressor = xgb.XGBRegressor(random_state=42)

param_grid = {"n_estimators": [100, 300, 500, 700, 900],
              "max_depth": [3, 5, 7, 9, 10],
              "learning_rate": [0.01, 0.05, 0.1]}

grid_search = GridSearchCV(estimator=xgb_regressor,
                           param_grid=param_grid,
                            scoring="neg_mean_squared_error",
                            verbose=1,
                            n_jobs=-1,
                            cv=pds)

grid_search.fit(x_combined, y_combined)

print("Best params:", grid_search.best_params_)

Fitting 1 folds for each of 75 candidates, totalling 75 fits
Best params: {'learning_rate': 0.05, 'max_depth': 9, 'n_estimators': 900}


In [59]:
model = xgb.XGBRegressor(n_estimators=900, learning_rate=0.05, max_depth=9, random_state=42)
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

model_one = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one

,r2,MAPE,MdAPE,MAE
Train Set,0.960678,6.699211,4.990321,$74923.56
Validation Set,0.908576,9.515434,6.829095,$110411.52
Test Set,0.917527,9.085087,6.470955,$108854.13


These are the best metrics we've seen yet, and this model has only been trained on the baseline dataset! This proves without a doubt that XGBoost is the optimal model for our dataset. For reference, the past r2 scores for this dataset have been 0.845539 (linear regression), 0.863098 (decision tree), and 0.902822 (random forest). Accounting for noise, XGBoost seems to be superior or at least on par with random forests when predicting our data.

## Model Two: Feature Engineered Dataset

In [3]:
# Read in the training and test sets created in week six

train_and_val = pd.read_csv("feat_eng_train.csv")
testing = pd.read_csv("feat_eng_test.csv")

# Carve out training and validation sets
training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265296, 153) (255004, 153) (10292, 153)


In [8]:
# Include the feature engineered columns BedBathRatio, PropertyAge, DistrictName_elem_enc, DistrictName_unif_enc,
# and DistrictName_high_enc. But don't include enrollment/demographic information for this test

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            "BedBathRatio", "PropertyAge"]

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [9]:
# Once again, create the same custom split for GridSearch validation

x_combined = np.vstack((x_train, x_val))
y_combined = np.concatenate((y_train, y_val))

split_indices = np.zeros(x_combined.shape[0])
split_indices[:len(x_train)] = -1 
split_indices[len(x_train):] = 0

pds = PredefinedSplit(test_fold=split_indices)

In [10]:
xgb_regressor = xgb.XGBRegressor(random_state=42)

param_grid = {"n_estimators": [100, 300, 500, 700, 900],
              "max_depth": [3, 5, 7, 9, 10],
              "learning_rate": [0.01, 0.05, 0.1]}

grid_search = GridSearchCV(estimator=xgb_regressor,
                           param_grid=param_grid,
                            scoring="neg_mean_squared_error",
                            verbose=1,
                            n_jobs=-1,
                            cv=pds)

grid_search.fit(x_combined, y_combined)

print("Best params:", grid_search.best_params_)

Fitting 1 folds for each of 75 candidates, totalling 75 fits
Best params: {'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 900}


In [11]:
model = xgb.XGBRegressor(n_estimators=900, learning_rate=0.05, max_depth=10, random_state=42)
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

model_two = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_two

,r2,MAPE,MdAPE,MAE
Train Set,0.975130,5.467347,4.071176,$60149.89
Validation Set,0.910982,9.337542,6.729750,$108562.68
Test Set,0.920399,8.875406,6.161915,$106030.26


There is a notable increase in performance for both the validation and test sets once the feature engineered dataset is used. This is a good sign that our added features are meaningful for predicting housing prices.

## Model Three: Add Demographic Data

The school district dataset that we joined with the MLS data contains a lot of demographic information. However, this is somewhat problematic for two reasons. On a performance level, the inclusion of these columns leads to a big leap in model complexity and makes the model much, much slower to run. On an ethical level, using racial demographic data to predict housing prices is very fraught, and can encode biases into the model. For this reason, this notebook will use only the non-racial demographic data included in the school district dataset.

In [5]:
common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            "BedBathRatio", "PropertyAge",  
            'EnrollTotal_high', 'EnrollCharter_high','EnrollNonCharter_high', # High school enrollment counts
            'EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem', # Elementary school enrollment counts
            'EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif', # Unified shcool district enrollment counts
            'ELpct_high','ELpct_elem','ELpct_unif', # English-learning student percent
            'FOSpct_high','FOSpct_elem','FOSpct_unif', # Foster student percent
            'HOMpct_high','HOMpct_elem','HOMpct_unif', # Homeless student percent
            'SEDpct_high', 'SEDpct_elem','SEDpct_unif'] # Socioeconomically disadvantaged student percent

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

x_combined = np.vstack((x_train, x_val))
y_combined = np.concatenate((y_train, y_val))

split_indices = np.zeros(x_combined.shape[0])
split_indices[:len(x_train)] = -1 
split_indices[len(x_train):] = 0

pds = PredefinedSplit(test_fold=split_indices)

In [6]:
xgb_regressor = xgb.XGBRegressor(random_state=42)

param_grid = {"n_estimators": [100, 300, 500, 700, 900],
              "max_depth": [3, 5, 7, 9, 10],
              "learning_rate": [0.01, 0.05, 0.1]}

grid_search = GridSearchCV(estimator=xgb_regressor,
                           param_grid=param_grid,
                            scoring="neg_mean_squared_error",
                            verbose=1,
                            n_jobs=-1,
                            cv=pds)

grid_search.fit(x_combined, y_combined)

print("Best params:", grid_search.best_params_)

Fitting 1 folds for each of 75 candidates, totalling 75 fits
Best params: {'learning_rate': 0.05, 'max_depth': 9, 'n_estimators': 900}


In [8]:
model = xgb.XGBRegressor(n_estimators=900, learning_rate=0.05, max_depth=10, random_state=42)
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

model_three = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_three

,r2,MAPE,MdAPE,MAE
Train Set,0.974545,5.523476,4.119242,$60949.42
Validation Set,0.911180,9.330702,6.664551,$108450.13
Test Set,0.920522,8.861699,6.194697,$105962.98


The performance of this model is slightly better. However, it's still very similar to the previous model, and for that reason the uptick in performance may be attributed simply to noise. This suggests that, for XGBoost at least, the addition of the demographic data does not substantially improve model performance.

## Summary

#### Summary of XGBoost models

| XGBoost Model | Test r2 | Test MAPE | Test MdAPE | Test MAE |
| ------------- | ------- | --------- | ----------| --------- |
| Model One (Baseline) | 0.917527 | 9.085087 | 6.470955 | \$108,854 |
| Model Two (Feature Engineering Additions) | 0.920399 | 8.875406 | 6.161915 | \$106,030 |
| **Model Two (Demographic Info Additions)** | **0.920522** | **8.861699** | **6.194697** | **\$105,962** |

#### Cross-Notebook Summary

| Model Type | Test r2 | Test MAPE | Test MdAPE | Test MAE |
| ---------- | ------- | --------- | ---------- | -------- |
| Linear Regression | 0.855653 | 12.550245 | 9.552526 | \$148,375 |
| Decision Tree | 0.867673 | 11.783563 | 8.438892 | \$140,170 |
| Random Forest | 0.905958 | 9.683137 | 6.796715 | \$116,191 |
| **XGBoost** | **0.920522** | **8.861699** | **6.194697** | **\$105,962** |

This notebook has shown that using XGBoost for predicting housing prices increases model performance on these standartd metrics significantly. The next step will be to look clsoer at what "performance" really means, with a more in-depth evaluation of this model.